# Tutorial 12-2: Breaking Boundaries with Multi-Layer ANNs
**Course:** CSEN 140: Data Mining/Machine Learning  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
We now solve the Sonar classification task using an **Artificial Neural Network (ANN)** with a hidden layer. We will demonstrate how adding internal 'neurons' allows the network to learn non-linear decision surfaces that a single Perceptron cannot.

### Theoretical Context
Training an ANN means learning the weights of multiple connected neurons. By using an **activation function** (like Sigmoid or ReLU) in the hidden layer, we introduce the non-linearity required to separate complex data. We will use **Xavier Initialization** to ensure weights are not set to zero, avoiding symmetry that prevents learning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load same Sonar data as 12-1
sonar = fetch_openml(name='sonar', version=1, as_frame=False)
le = LabelEncoder()
y = le.fit_transform(sonar.target)
scaler = StandardScaler()
X = scaler.fit_transform(sonar.data)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = torch.FloatTensor(X_train), torch.FloatTensor(X_test)
y_train, y_test = torch.FloatTensor(y_train).view(-1, 1), torch.FloatTensor(y_test).view(-1, 1)

## 2. Architecture: Input, Hidden, and Output Layers
Following the notation from the slides, our model features an input layer ($x$), a hidden layer ($a^{[2]}$), and an output layer ($a^{[3]}$).

In [ ]:
class MultiLayerANN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(MultiLayerANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
        # Xavier/He Initialization
        nn.init.kaiming_uniform_(self.layer1.weight, nonlinearity='relu')
        nn.init.xavier_uniform_(self.layer2.weight)
        
    def forward(self, x):
        # Forward Propagation
        a2 = self.relu(self.layer1(x))
        a3 = self.sigmoid(self.layer2(a2))
        return a3

model = MultiLayerANN(X_train.shape[1], 24)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

## 3. Training and Evaluation
We use Backpropagation to update the weights based on the error $\delta$ of each node.

In [ ]:
for epoch in range(200):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_acc = (model(X_test).round().eq(y_test).sum() / float(len(y_test))).item()
    print(f"ANN Multi-Layer Test Accuracy: {test_acc:.4f}")

## Conclusion
On the Sonar data, the **Multi-Layer ANN** outperforms the tuned linear model. The hidden layer acts as a 'feature engineer,' allowing the model to project inputs into a space where they become separable. This transition from simple sums to layered hierarchies is what powers Deep Learning.